# ⚡ Snippy (Google Colab Version): Gemma 3 270M Fine-Tuning & LiteRT Export

This notebook is specifically tailored for **Google Colab (T4 GPU / CUDA 13 Driver / Python 3.12)**.
It fine-tunes **Gemma 3 270M** as **Snippy** — an in-browser JavaScript Code Snippet & Tool Calling Agent — and exports the model to Google LiteRT format for browser WebGPU deployment.

## Step 1: Environment Setup
We set `USE_TORCHVISION=0` to bypass Colab `torchvision` C++ operator mismatches, and install ML + LiteRT dependencies.

In [ ]:
import os
# Disable torchvision imports to prevent C++ operator ABI mismatches on Colab CUDA driver
os.environ["USE_TORCHVISION"] = "0"

# Install PyTorch, Transformers, PEFT, TRL, Datasets, and LiteRT Tools
!pip install -q -U torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -q -U transformers peft trl datasets litert-torch litert-lm

## Step 2: Fetch Snippy Training Dataset
Fetch `snippy_dataset.json` containing 50+ generic tool calling instruction examples.

In [ ]:
import json
import os
from datasets import Dataset

DATASET_PATH = "snippy_dataset.json"
if not os.path.exists(DATASET_PATH):
    !wget -q https://raw.githubusercontent.com/silasly/gemma-litert-snippy-demo/main/snippy_dataset.json

with open(DATASET_PATH, "r") as f:
    sample_data = json.load(f)

dataset = Dataset.from_list(sample_data)
print(f"✅ Loaded {len(dataset)} training examples for Snippy in Colab!")

## Step 3: Fine-Tune Gemma 3 270M with PEFT / LoRA
Train Gemma 3 270M on Colab T4 GPU.

In [ ]:
import os
os.environ["USE_TORCHVISION"] = "0"

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig

MODEL_ID = "unsloth/gemma-3-270m-it"
OUTPUT_LORA_DIR = "./lora_adapter"
OUTPUT_MERGED_DIR = "./fine_tuned_gemma_merged"

# 1. Load Tokenizer & Base Model
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

# 2. Configure LoRA for Gemma 3
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
peft_model = get_peft_model(model, peft_config)

# 3. Configure Trainer (loss_type='nll' avoids forward patch error)
sft_config = SFTConfig(
    dataset_text_field="text",
    max_length=256,
    output_dir="./results",
    num_train_epochs=5,
    per_device_train_batch_size=2,
    logging_steps=1,
    loss_type="nll"
)
trainer = SFTTrainer(
    model=peft_model,
    train_dataset=dataset,
    args=sft_config
)
trainer.train()

# Save LoRA Adapter
peft_model.save_pretrained(OUTPUT_LORA_DIR)
tokenizer.save_pretrained(OUTPUT_LORA_DIR)
print("✅ LoRA Adapter Saved in Colab!")

## Step 4: Merge Adapter Weights into Base Model
Merge LoRA adapter weights into base weights.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch

base_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float32, device_map="cpu")
peft_model = PeftModel.from_pretrained(base_model, OUTPUT_LORA_DIR)
merged_model = peft_model.merge_and_unload()

merged_model.save_pretrained(OUTPUT_MERGED_DIR)
tokenizer.save_pretrained(OUTPUT_MERGED_DIR)
print("✅ Merged Model Saved!")

## Step 5: Convert Model to LiteRT Format (`litert-torch export_hf`)
Export model to `.litertlm` container with INT8 dynamic quantization.

In [ ]:
!litert-torch export_hf \
  ./fine_tuned_gemma_merged \
  ./litert_output \
  -b True \
  -q dynamic_int8

## Step 6: Extract WebGPU FlatBuffer & Download Converted Models
Extract the `TFL3` FlatBuffer (`model.tflite`) for WebGPU browser runtime and trigger direct file downloads in Colab.

In [ ]:
import os
from google.colab import files

SOURCE_MODEL = "./litert_output/model.litertlm"

if os.path.exists(SOURCE_MODEL):
    # Extract TFLite FlatBuffer for WebGPU browser engine
    with open(SOURCE_MODEL, "rb") as f:
        data = f.read()
    pos = data.find(b"TFL3")
    if pos != -1:
        tflite_data = data[pos - 4 :]
        with open("model.tflite", "wb") as f:
            f.write(tflite_data)
        print("✅ Extracted model.tflite (WebGPU FlatBuffer)!")
        files.download("model.tflite")
    
    print("✅ Downloading model.litertlm...")
    files.download(SOURCE_MODEL)
else:
    print(f"⚠️ Source model not found at {SOURCE_MODEL}")